Let's create a document

In [1]:
document = """
Python generators are a special type of iterator that allow values to be
produced lazily. Instead of creating and storing all values in memory at once,
a generator produces each value only when it is requested.

Generators are commonly created using functions containing the yield keyword.
When a generator function is called, Python returns a generator object rather
than immediately executing the entire function.

For example, a generator can produce numbers from 1 to 5 one at a time.
The yield statement pauses execution of the function and remembers its state.
When the next value is requested, execution continues from where it previously
stopped.

Generators are useful when working with large datasets or streams because they
avoid loading all values into memory at the same time. This can significantly
reduce memory usage.

Python also provides generator expressions, which have syntax similar to list
comprehensions but produce values lazily instead of constructing the entire
collection immediately.

Generators can be consumed using iteration, such as a for loop, or by calling
the next function to request individual values.
"""

In [2]:
print(len(document))

1141


The simplest possible chunking

In [3]:
chunks = [
    document[:300],
    document[300:600],
    document[600:900],
    document[900:]
]

for i, chunk in enumerate(chunks):
  print(f"\n--- Chunk {i+1} ---")
  print(chunk)


--- Chunk 1 ---

Python generators are a special type of iterator that allow values to be
produced lazily. Instead of creating and storing all values in memory at once,
a generator produces each value only when it is requested.

Generators are commonly created using functions containing the yield keyword.
When a ge

--- Chunk 2 ---
nerator function is called, Python returns a generator object rather
than immediately executing the entire function.

For example, a generator can produce numbers from 1 to 5 one at a time.
The yield statement pauses execution of the function and remembers its state.
When the next value is requested

--- Chunk 3 ---
, execution continues from where it previously
stopped.

Generators are useful when working with large datasets or streams because they
avoid loading all values into memory at the same time. This can significantly
reduce memory usage.

Python also provides generator expressions, which have syntax si

--- Chunk 4 ---
milar to list
comprehensions 

Install LangChain text splitters

In [4]:
!pip -q install langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(document)

print("Number of chunks:", len(chunks))

Number of chunks: 6


In [6]:
for i, chunk in enumerate(chunks):
  print(f"\n--- Chunk {i+1} ---")
  print(chunk)


--- Chunk 1 ---
Python generators are a special type of iterator that allow values to be
produced lazily. Instead of creating and storing all values in memory at once,
a generator produces each value only when it is requested.

--- Chunk 2 ---
Generators are commonly created using functions containing the yield keyword.
When a generator function is called, Python returns a generator object rather
than immediately executing the entire function.

--- Chunk 3 ---
For example, a generator can produce numbers from 1 to 5 one at a time.
The yield statement pauses execution of the function and remembers its state.
When the next value is requested, execution continues from where it previously
stopped.

--- Chunk 4 ---
Generators are useful when working with large datasets or streams because they
avoid loading all values into memory at the same time. This can significantly
reduce memory usage.

--- Chunk 5 ---
Python also provides generator expressions, which have syntax similar to list
compre

Recursive is used to preserve meaningful boundries before resorting to smaller ones.

First experiment

In [8]:
for size in [100, 300, 500, 1000]:

  splitter = RecursiveCharacterTextSplitter(
      chunk_size=size,
      chunk_overlap=50
  )

  chunks = splitter.split_text(document)

  print(
      f"chunk_size={size}",
      f" -> {len(chunks)} chunks"
  )

chunk_size=100  -> 15 chunks
chunk_size=300  -> 6 chunks
chunk_size=500  -> 3 chunks
chunk_size=1000  -> 2 chunks


Retrieval quality

In [9]:
questions = [
    "How do Python generators save memory?",
    "What keyword is used to create generator functions?",
    "How can I request one value from a generator?",
    "What is a generator expression?"
]

Three major chunking strategies

1. Fixed-size chunking
2. Recursive chunking
3. Semantic chunking

In [10]:
query = "How do generators reduce memory usage?"

In [11]:
splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

splitter_medium = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

In [12]:
small_chunks = splitter_small.split_text(document)
medium_chunks = splitter_medium.split_text(document)
large_chunks = splitter_large.split_text(document)

In [14]:
print("Small:", len(small_chunks))
print("Medium:", len(medium_chunks))
print("Large:", len(large_chunks))

Small: 15
Medium: 6
Large: 2


Create embeddings for each

In [16]:
!pip -q install sentence-transformers

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
small_embeddings = model.encode(small_chunks)
medium_embeddimgs = model.encode(medium_chunks)
large_embeddings = model.encode(large_chunks)

Create three Chroma collections

In [21]:
!pip -q install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [22]:
import chromadb

client  = chromadb.Client()

small_collection = client.create_collection(
    name = "small_chunks"
)

medium_collection = client.create_collection(
    name = "medium_chunks"
)

large_collection = client.create_collection(
    name = "large_chunks"
)

In [23]:
small_collection.add(
    ids = [f"small_{i}" for i in range(len(small_chunks))],
    documents = small_chunks,
    embeddings = small_embeddings.tolist()
)

medium_collection.add(
    ids = [f"medium_{i}" for i in range(len(medium_chunks))],
    documents = medium_chunks,
    embeddings = medium_embeddimgs.tolist()
)

large_collection.add(
    ids = [f"large_{i}" for i in range(len(large_chunks))],
    documents = large_chunks,
    embeddings = large_embeddings.tolist()
)

Ask one question

In [24]:
query = "How do generators reduce memory usage?"

In [27]:
query_embedding = model.encode([query])[0]

Search each collection

In [31]:
small_results = small_collection.query(
    query_embeddings = [query_embedding.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

medium_results = medium_collection.query(
    query_embeddings = [query_embedding.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

large_results = large_collection.query(
    query_embeddings = [query_embedding.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

Create a helper to display them

In [32]:
def print_results(name,results):
  print(f"\n{'='*20}")
  print(name)
  print(f"{'='*20}")

  for i, (doc,distance) in enumerate(
      zip(
          results["documents"][0],
          results["distances"][0]
      ),
      start=1
  ):
    print(f"\nResult {i}")
    print(f"Distance: {distance:.4f}")
    print(doc)

In [33]:
print_results("SMALL CHUNKS", small_results)
print_results("MEDIUM CHUNKS", medium_results)
print_results("LARGE CHUNKS", large_results)


SMALL CHUNKS

Result 1
Distance: 0.6860
Generators can be consumed using iteration, such as a for loop, or by calling

Result 2
Distance: 0.6866
Generators are useful when working with large datasets or streams because they

Result 3
Distance: 0.8455
Generators are commonly created using functions containing the yield keyword.

MEDIUM CHUNKS

Result 1
Distance: 0.4694
Generators are useful when working with large datasets or streams because they
avoid loading all values into memory at the same time. This can significantly
reduce memory usage.

Result 2
Distance: 0.8123
Python generators are a special type of iterator that allow values to be
produced lazily. Instead of creating and storing all values in memory at once,
a generator produces each value only when it is requested.

Result 3
Distance: 0.8139
Generators can be consumed using iteration, such as a for loop, or by calling
the next function to request individual values.

LARGE CHUNKS

Result 1
Distance: 0.6681
Generators are use

Let's test another question

In [34]:
query = "What keyword is used to create a Python generator?"

In [35]:
query_embedding = model.encode([query])[0]

for name, collection in [
    ("SMALL", small_collection),
    ("MEDIUM", medium_collection),
    ("LARGE", large_collection)
]:

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=2,
        include=["documents", "distances"]
    )

    print_results(name, results)


SMALL

Result 1
Distance: 0.5640
When a generator function is called, Python returns a generator object rather

Result 2
Distance: 0.5776
Generators are commonly created using functions containing the yield keyword.

MEDIUM

Result 1
Distance: 0.3452
Generators are commonly created using functions containing the yield keyword.
When a generator function is called, Python returns a generator object rather
than immediately executing the entire function.

Result 2
Distance: 0.4666
Python generators are a special type of iterator that allow values to be
produced lazily. Instead of creating and storing all values in memory at once,
a generator produces each value only when it is requested.

LARGE

Result 1
Distance: 0.4292
Python generators are a special type of iterator that allow values to be
produced lazily. Instead of creating and storing all values in memory at once,
a generator produces each value only when it is requested.

Generators are commonly created using functions containing t